In [1]:
from datasets import load_dataset

c:\Users\PC\.virtualenvs\deep-learning-1I3A7gMi\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
dataset = load_dataset('imdb')
small_train = dataset['train'].shuffle(seed=42).select([i for i in list(range(1000))])
small_test = dataset['test'].shuffle(seed=42).select([i for i in list(range(300))])

In [3]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')

def preprocess_function(examples):
    return tokenizer(examples['text'], truncation=True, padding='max_length', max_length=256)

tokenized_train = small_train.map(preprocess_function, batched=True)
tokenized_test = small_train.map(preprocess_function, batched=True)


In [4]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=2)

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 480.18it/s, Materializing param=distilbert.transformer.layer.5.sa_layer_norm.weight]   
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [5]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir = './results',
    num_train_epochs = 3,
    per_device_train_batch_size = 16,
    per_device_eval_batch_size = 16,
    logging_dir = './logs',
    logging_steps = 10,
    report_to = 'none'
)

trainer = Trainer(
    model = model, 
    args = training_args,
    train_dataset = tokenized_train,
    eval_dataset = tokenized_test
)

trainer.train()

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.
c:\Users\PC\.virtualenvs\deep-learning-1I3A7gMi\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
10,0.711984
20,0.674071
30,0.679998
40,0.648531
50,0.453370
60,0.365898
70,0.555626
80,0.304614
90,0.334693
100,0.368143


Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.47s/it]


TrainOutput(global_step=189, training_loss=0.370489274383222, metrics={'train_runtime': 2057.7822, 'train_samples_per_second': 1.458, 'train_steps_per_second': 0.092, 'total_flos': 198701097984000.0, 'train_loss': 0.370489274383222, 'epoch': 3.0})